# MSH 3-T-junction array: logical gates from the tensor-network evolution (reduced method)

The gates of `msh_tjunctions_tn.ipynb` (√Z and Z from `exchange_12`, √X and X from `exchange_23` with `rot_Tx = 400`) as **logical
unitaries** on the qubit $|0\rangle_L = |00\rangle$, $|1\rangle_L = |11\rangle$, with signs and relative phases. This is the *reduced method* of the
earlier `msh_pfaffian` work: only the two zero-mode fermions $\psi_A$, $\psi_B$ are evolved (here with MPS/TDVP), and the gate follows from
the $4\times4$ real Majorana matrix
$$M_{ij} = \langle\gamma_i(0)|S_T\,\gamma_j(0)\rangle .$$
It is the generalized Wick (Pfaffian) calculation restricted to the zero-mode block, so it is exact when the bulk stays unexcited; the
full calculation over all $2N$ occupied quasiparticles, which also gives the probability of exciting the bulk, is in
`logical_gates_python_ed.ipynb` (dense), and that notebook compares the two.

**No new time evolution.** `msh_tjunctions_tn.ipynb` (section 6) records $z_{ik}(t) = \langle\gamma_i(0)|\psi_k(t)\rangle$ at every snapshot,
including the ends of the first and second exchange, and caches them in `msh_tjunctions_tdvp_{Z,X}.jls`. With
$\psi_k = (\gamma_{2k-1} + i\gamma_{2k})/\sqrt2$, self-conjugate Majoranas ($C\gamma = \gamma$) and $[S_T, C] = 0$,
$$\gamma_{2k-1}(T) = \frac{\psi_k(T) + C\psi_k(T)}{\sqrt2},\qquad \gamma_{2k}(T) = \frac{\psi_k(T) - C\psi_k(T)}{i\sqrt2},\qquad
\langle\gamma_i|C\psi\rangle = \langle\psi|\gamma_i\rangle = z_{ik}^*,$$
so
$$M_{i,2k-1} = \sqrt2\,\mathrm{Re}\,z_{ik},\qquad M_{i,2k} = \sqrt2\,\mathrm{Im}\,z_{ik}.$$
Everything below is $4\times4$ algebra on these numbers; the notebook runs in seconds. The caches are not in the repository
(`*.jls`); running `msh_tjunctions_tn.ipynb` creates them (about 2.5 hours of TDVP with six threads).

**Contents**
1. Setup and the cached overlaps
2. From the Majorana matrix to the logical gate
3. Logical gates after one and two exchanges
4. Export (`logical_gates_tn.csv`)

## 1. Setup and the cached overlaps

In [1]:
using LinearAlgebra, Printf, Serialization

# gate durations, as `duration` in msh_tjunctions_tn.ipynb (section 2): one exchange_12 (rot_Tz = 200), one exchange_23 (rot_Tx = 400)
const L1, L2 = 14, 14
const T1 = Dict("Z" => (0.1 * (4L1 + 2L2 + 8) + 1) * 200.0, "X" => (0.1 * (L1 + 3L2 + 2L1 + 3) + 1) * 400.0)

cache(kind) = "msh_tjunctions_tdvp_$kind.jls"
for kind in ("Z", "X")
    isfile(cache(kind)) || error("$(cache(kind)) not found: run msh_tjunctions_tn.ipynb (section 6) in this directory first")
end
runs = Dict(kind => deserialize(cache(kind)) for kind in ("Z", "X"))

"Overlaps z[i, k] = ⟨γ_i(0)|ψ_k(t)⟩ (4 × 2) of run `kind` at the snapshot t."
function overlaps(kind, t)
    d = runs[kind]; i = findfirst(≈(t), d.ts)
    i === nothing && error("no snapshot at t = $t in $(cache(kind))")
    return hcat(d.z[1][:, i], d.z[2][:, i])
end
for kind in ("Z", "X")
    @printf("%s: one exchange T₁ = %.0f, snapshots up to t = %.0f\n", kind, T1[kind], runs[kind].ts[end])
end

Z: one exchange T₁ = 2040, snapshots up to t = 4460
X: one exchange T₁ = 3880, snapshots up to t = 8140


## 2. From the Majorana matrix to the logical gate

* `braid_matrix(z)`: $M$ from the overlaps (derivation in the introduction); $\|M^TM - 1\|$ measures the leakage out of the zero-mode subspace.
* Restricted to the zero-mode Fock space (bulk unexcited), $\hat U$ acts as the unitary $\tilde U$ with
  $\tilde U\hat\gamma_j\tilde U^\dagger = \sum_i\hat\gamma_i M_{ij}$. With $M = e^A$ ($A$ real antisymmetric),
  $\tilde U = \exp(\tfrac14\sum_{ij}A_{ij}\hat\gamma_i\hat\gamma_j)$, up to a global phase and sign (double cover of SO(4)).
* `jw_majoranas`: the four $\hat\gamma_i$ as $4\times4$ matrices on $|n_An_B\rangle$ (order 00, 01, 10, 11) via Jordan–Wigner,
  $f_A = a\otimes 1$, $f_B = Z\otimes a$, with $f_m$ the annihilator of the **positive-energy** mode $\psi_m = (\gamma_{2m-1} + i\gamma_{2m})/\sqrt2$, so that
  $n_m = 1$ means $\psi_m$ occupied, as in the masks of the CZ notebooks. The quasiparticle operator $b(w) = w^\dagger\Psi$ is antilinear in $w$, so
  $b(\psi_m) = (\hat\gamma_{2m-1} - i\hat\gamma_{2m})/\sqrt2$ and hence $\hat\gamma_{2m-1} = f_m + f_m^\dagger$, $\hat\gamma_{2m} = +i(f_m - f_m^\dagger)$.
  (The opposite sign, used in the earlier `msh_logical_gate.jl`, labels $|00\rangle$ and $|11\rangle$ the other way round and conjugates every
  relative phase; `logical_gates_python_ed.ipynb` checks the convention against the full Pfaffian amplitudes entry by entry.)
* `logical_unitary`: replaces $M$ by its nearest orthogonal matrix (polar decomposition, i.e. renormalizes away leakage), takes
  $A = \log M$, and picks the sign of $A$ for which $\tilde U\hat\gamma\tilde U^\dagger = \sum\hat\gamma M$ holds (the residual is returned).
* `logical_even_block`: the even-sector block $U_L$ on $\{|00\rangle, |11\rangle\}$ (the logical qubit), described in two complementary ways:
    * **Rotation angle $\theta$ and axis $\hat n$ — not phases.** With the global phase removed,
      $U_L/\sqrt{\det U_L} = \cos\frac\theta2 - i\sin\frac\theta2\,\hat n\cdot\boldsymbol\sigma = e^{-i\theta\,\hat n\cdot\boldsymbol\sigma/2}$: the gate rotates
      the logical Bloch sphere by the angle $\theta$ about the axis $\hat n$. Ideal values: $\sqrt X = e^{-i\pi X/4}$ has $\theta = 90°$, $\hat n = \hat x$
      ($\sqrt X^\dagger$: $\hat n = -\hat x$); $X$ has $\theta = 180°$; a diagonal gate $\mathrm{diag}(1, e^{i\varphi})$ has $\theta = |\varphi|$, $\hat n = \mp\hat z$.
      The amplitudes follow from it: $P(00\to11) = \sin^2\frac\theta2\,(1 - \hat n_z^2)$. $(\theta, \hat n)$ and $(360° - \theta, -\hat n)$ are the same
      gate up to a global sign; the code reports the pair given by the principal $\sqrt{\det U_L}$.
    * **Relative phases** of the matrix elements, independent of the global phase:
      $\varphi_d = \arg\langle11|\hat U|11\rangle - \arg\langle00|\hat U|00\rangle$ between the diagonal elements (the gate phase of a diagonal gate:
      $-90°$ for $\sqrt Z^\dagger$, $180°$ for $Z$), and $\varphi_o = \arg\langle11|\hat U|00\rangle - \arg\langle00|\hat U|11\rangle$ between the off-diagonal
      elements. For a rotation about an axis in the $xy$ plane at azimuth $\phi$, $\varphi_o = 2\phi$: it is $0$ for every rotation about $\pm\hat x$
      ($\sqrt X$, $\sqrt X^\dagger$, $X$, any $\theta$) and $180°$ about $\pm\hat y$. The phase of an off-diagonal element *relative to a diagonal one*,
      $\arg\langle11|\hat U|00\rangle - \arg\langle00|\hat U|00\rangle$, is $\pm90°$ for a rotation about $\hat x$, but its sign is that of $\cos\frac\theta2$: it
      flips when the rotation passes $180°$ (two over-rotated exchanges, $2\times92.53° = 185.06°$) and is undefined for an exact $X$, whose diagonal
      vanishes. It is therefore not used.
    * The global-phase-insensitive overlaps $|\mathrm{Tr}(V^\dagger U_L)|/2$ with the ideal gates $V$.

Ideal gates in the even sector: one `exchange_12` (γ₁ ↔ γ₂, the same fermion) gives $\mathrm{diag}(1, e^{\pm i\pi/2})$, i.e. $\sqrt Z$ up to its
sign convention; two give $Z$. One `exchange_23` (γ₂ ↔ γ₃, different fermions) gives $\sqrt X$ or $\sqrt X^\dagger$; two give $X$.
The sign of each exchange ($\pm$) depends on the orientation of the braid and the sign conventions of the $\gamma$'s, so both are listed.

In [2]:
"M[i, j] = ⟨γ_i(0)|γ_j(T)⟩ from z[i, k] = ⟨γ_i(0)|ψ_k(T)⟩:  M[:, 2k−1] = √2 Re z[:, k],  M[:, 2k] = √2 Im z[:, k]."
braid_matrix(z) = reduce(hcat, [[√2 .* real.(z[:, k]) √2 .* imag.(z[:, k])] for k in 1:size(z, 2)])

function jw_majoranas()
    a = ComplexF64[0 1; 0 0]                       # annihilator, basis (|0⟩, |1⟩)
    Z = ComplexF64[1 0; 0 -1]; I2 = Matrix{ComplexF64}(I, 2, 2)
    fA, fB = kron(a, I2), kron(Z, a)
    return [fA + fA', im * (fA - fA'), fB + fB', im * (fB - fB')]   # f_m annihilates ψ_m = (γ_{2m−1} + iγ_{2m})/√2
end

"Spin representation Ũ of the Majorana rotation M on the zero-mode Fock space (order 00, 01, 10, 11), and its residual."
function logical_unitary(M)
    F = svd(real.(M)); Mo = F.U * F.Vt                  # nearest orthogonal matrix
    det(Mo) > 0 || error("M has det −1: not a parity-preserving rotation")
    γ = jw_majoranas()
    best = nothing
    for sgn in (1, -1)
        A = real.(log(Mo)); A = (A - A') / 2
        G = sum(sgn * A[i, j] * γ[i] * γ[j] for i in 1:4, j in 1:4) / 4
        U = exp(G)
        res = maximum(norm(U * γ[j] * U' - sum(γ[i] * Mo[i, j] for i in 1:4)) for j in 1:4)
        (best === nothing || res < best[2]) && (best = (U, res))
    end
    return best
end

const σL = (ComplexF64[0 1; 1 0], ComplexF64[0 -im; im 0], ComplexF64[1 0; 0 -1])
const IDEAL = ["√X" => exp(-im * π / 4 * σL[1]), "√X†" => exp(im * π / 4 * σL[1]), "X" => σL[1],
               "√Z" => ComplexF64[1 0; 0 im], "√Z†" => ComplexF64[1 0; 0 -im], "Z" => σL[3]]

"""
Even-sector block U_L on {|00⟩, |11⟩}: the Bloch-sphere rotation angle θ and axis n̂ (U_L ∝ exp(−iθ n̂·σ/2); not phases),
P(00 → 11), the relative phases φd = arg U_L[2,2] − arg U_L[1,1] and φo = arg U_L[2,1] − arg U_L[1,2] (degrees; NaN where the elements
vanish), and the overlaps |Tr(V†U_L)|/2 with the ideal gates.
"""
function logical_even_block(Ũ)
    UL = Ũ[[1, 4], [1, 4]]
    W = UL / sqrt(det(UL))                              # cos(θ/2) − i sin(θ/2) n̂·σ
    θ = 2acos(clamp(real(tr(W)) / 2, -1, 1))
    n̂ = sin(θ / 2) < 1e-12 ? zeros(3) : [real(im * tr(σ * W)) / (2sin(θ / 2)) for σ in σL]
    φd = min(abs(UL[1, 1]), abs(UL[2, 2])) > 1e-6 ? rad2deg(angle(UL[2, 2] / UL[1, 1])) : NaN
    φo = min(abs(UL[2, 1]), abs(UL[1, 2])) > 1e-6 ? rad2deg(angle(UL[2, 1] / UL[1, 2])) : NaN
    F = Dict(name => abs(tr(V' * UL)) / 2 for (name, V) in IDEAL)
    return (; UL, θ, n̂, P01 = abs2(UL[2, 1]), φd, φo, F)
end
deg(x) = isnan(x) ? "—" : @sprintf("%+.4f°", x)

# check: an exact braid γ₂ → γ₃, γ₃ → −γ₂ (others fixed) must give √X or √X†
Mb = Matrix(1.0I, 4, 4); Mb[2, 2] = Mb[3, 3] = 0; Mb[3, 2] = 1; Mb[2, 3] = -1
Ub, res = logical_unitary(Mb); Lb = logical_even_block(Ub)
@printf("ideal γ₂ ↔ γ₃ braid: residual %.1e, θ = %.4f°, n̂ = %s, √X %.6f, √X† %.6f\n", res, rad2deg(Lb.θ),
        string(round.(Lb.n̂, digits=6)), Lb.F["√X"], Lb.F["√X†"])

ideal γ₂ ↔ γ₃ braid: residual 8.0e-16, θ = 90.0000°, n̂ = [-1.0, 0.0, 0.0], √X 0.000000, √X† 1.000000


## 3. Logical gates after one and two exchanges

For each run, $M$ at the end of the first exchange ($t = T_1$: √Z or √X) and of the second ($t = 2T_1$: Z or X). Printed: $M$, the leakage of
$\psi_A$, $\psi_B$ ($1 - \sum_i|z_{ik}|^2$) and $\|M^TM - 1\|$, the transition probability $P(00\to11)$ with the check
$\sin^2\frac\theta2(1 - \hat n_z^2)$, the Bloch-sphere rotation angle $\theta$ and axis $\hat n$, the relative phases $\varphi_d$ (diagonal) and $\varphi_o$
(off-diagonal), and the overlaps with the ideal gates (section 2 defines all of them).

In [3]:
results = []
for (kind, stage, t) in [("Z", 1, T1["Z"]), ("Z", 2, 2T1["Z"]), ("X", 1, T1["X"]), ("X", 2, 2T1["X"])]
    z = overlaps(kind, t)
    M = braid_matrix(z)
    leak = [1 - sum(abs2, z[:, k]) for k in 1:2]
    Ũ, res = logical_unitary(M)
    L = logical_even_block(Ũ)
    name = kind == "Z" ? (stage == 1 ? "√Z (exchange_12)" : "Z (exchange_12 × 2)") : (stage == 1 ? "√X (exchange_23)" : "X (exchange_23 × 2)")
    println("── ", name, @sprintf(",  t = %.0f", t))
    show(stdout, "text/plain", round.(M, digits=6)); println()
    @printf("   leak (ψ_A, ψ_B) = %s,  ‖MᵀM − 1‖ = %.1e,  spin-representation residual %.1e\n",
            string(round.(leak, sigdigits=3)), norm(M' * M - I), res)
    @printf("   P(00→11) = %.6f  (sin²(θ/2)(1 − n̂_z²) = %.6f);  rotation θ = %.4f° about n̂ = %s\n", L.P01,
            max(0.0, sin(L.θ / 2)^2 * (1 - L.n̂[3]^2)), rad2deg(L.θ), string(round.(L.n̂, digits=4)))
    println("   relative phases: φd = arg⟨11|Û|11⟩ − arg⟨00|Û|00⟩ = ", deg(L.φd), ",  φo = arg⟨11|Û|00⟩ − arg⟨00|Û|11⟩ = ", deg(L.φo))
    println("   |Tr(V†U_L)|/2: ", join([@sprintf("%s %.6f", n, L.F[n]) for (n, _) in IDEAL], ",  "))
    push!(results, (; kind, stage, t, M, leak, L))
end

── √Z (exchange_12),  t = 2040
4×4 Matrix{Float64}:
 -0.012528  -0.999921  0.0     -0.0
  0.999921  -0.012528  0.0     -0.0
  0.0       -0.0       1.0     -2.0e-6
  0.0       -0.0       2.0e-6   1.0
   leak (ψ_A, ψ_B) = [9.72e-7, 1.65e-11],  ‖MᵀM − 1‖ = 1.4e-06,  spin-representation residual 2.6e-15
   P(00→11) = 0.000000  (sin²(θ/2)(1 − n̂_z²) = 0.000000);  rotation θ = 90.7180° about n̂ = [-0.0, -0.0, -1.0]
   relative phases: φd = arg⟨11|Û|11⟩ − arg⟨00|Û|00⟩ = -90.7180°,  φo = arg⟨11|Û|00⟩ − arg⟨00|Û|11⟩ = —
   |Tr(V†U_L)|/2: √X 0.496858,  √X† 0.496858,  X 0.000000,  √Z 0.006265,  √Z† 0.999980,  Z 0.711523
── Z (exchange_12 × 2),  t = 4080
4×4 Matrix{Float64}:
 -0.999685   0.025054  -0.0      0.0
 -0.025054  -0.999685  -0.0     -0.0
 -0.0       -0.0        1.0     -5.0e-6
 -0.0       -0.0        5.0e-6   1.0
   leak (ψ_A, ψ_B) = [1.99e-6, 2.07e-11],  ‖MᵀM − 1‖ = 2.8e-06,  spin-representation residual 1.2e-15
   P(00→11) = 0.000000  (sin²(θ/2)(1 − n̂_z²) = 0.000000);  rotation θ = 17

## 4. Export

`logical_gates_tn.csv`: one row per gate and stage with $t$, the leakage of both modes and the 16 entries of $M$ (row-major), read by
`logical_gates_python_ed.ipynb` for the comparison with the dense calculation.

In [4]:
open("logical_gates_tn.csv", "w") do io
    println(io, join(vcat(["gate", "stage", "t", "leak_A", "leak_B"], ["M$i$j" for i in 1:4 for j in 1:4]), ","))
    for r in results
        println(io, join(vcat([r.kind, r.stage, r.t, r.leak...], [r.M[i, j] for i in 1:4 for j in 1:4]), ","))
    end
end
println("wrote logical_gates_tn.csv (", length(results), " rows)")

wrote logical_gates_tn.csv (4 rows)
